In [1]:
# %%
# Banana subset — models A, B, and A+B
# T3/T4 = T7/T8 in modern 10-20 notation
from __future__ import annotations

import os
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
)
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

import mne
mne.set_log_level("WARNING")

In [2]:
# %%
PROJECT_ROOT  = Path("..").resolve()
DERIVED_ROOT  = PROJECT_ROOT / "data" / "derived"
MANIFEST_PATH = DERIVED_ROOT / "manifests" / "manifest_spontaneous_validated.csv"
FEAT_A_PATH   = DERIVED_ROOT / "features" / "features_A_bandpower_epochwise.csv"
MATRICES_PATH = DERIVED_ROOT / "features" / "features_B_wpli_matrices_epoch_sliding.npz"
FULL_RESULTS_PATH = PROJECT_ROOT / "results" / "models" / "results_rf_nested_groupkfold_Aepoch_BwpliEpochSliding.csv"
BANANA_B_PATH = PROJECT_ROOT / "results" / "models" / "results_rf_banana_wpli.csv"
OUT_DIR       = PROJECT_ROOT / "results" / "models"
FIGURE_DIR    = PROJECT_ROOT / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

BANANA_CHANNELS = ["O1", "O2", "T7", "T8"]   # T3=T7, T4=T8
BANDS_BP        = {"delta": (1.0, 4.0), "theta": (4.0, 8.0),
                   "alpha": (8.0, 13.0), "beta": (13.0, 30.0)}
BANDS_WPLI      = ["theta", "alpha", "beta"]
REJECT_PTP_UV   = 250.0
EYES_KEEP       = "closed"

OUTER_SPLITS = 5
INNER_SPLITS = 4
N_ITER       = 50
SEED         = 0
N_CORES      = os.cpu_count() or 8
print(f"Cores: {N_CORES}")

Cores: 60


In [3]:
# %%
# ============================================
# Section 1. Channel metadata
# ============================================

manifest = pd.read_csv(MANIFEST_PATH)
sample_epochs = mne.io.read_epochs_eeglab(manifest["file_path"].iloc[0], verbose="ERROR")
ch_names  = sample_epochs.ch_names
ch_info   = sample_epochs.info

banana_idx = [ch_names.index(c) for c in BANANA_CHANNELS]
n_banana   = len(banana_idx)
triu_r, triu_c = np.triu_indices(n_banana, k=1)
wpli_labels = [
    f"{band}_{BANANA_CHANNELS[r]}-{BANANA_CHANNELS[c]}"
    for band in BANDS_WPLI for r, c in zip(triu_r, triu_c)
]
bp_labels = [
    f"bp_{band}_{ch}"
    for band in BANDS_BP for ch in BANANA_CHANNELS
]
print(f"wPLI features: {len(wpli_labels)}  |  bandpower features: {len(bp_labels)}")

wPLI features: 18  |  bandpower features: 16


In [4]:
# %%
# ============================================
# Section 2a. Extract banana bandpower (A) from raw .set files
# 4 channels × 4 bands = 16 features per epoch
# ============================================

df_manifest_ec = manifest[manifest["eyes"] == EYES_KEEP].copy()
df_manifest_ec = df_manifest_ec.sort_values(["subject_id", "recording_number"]).reset_index(drop=True)

bp_rows = []
for _, row in df_manifest_ec.iterrows():
    epochs = mne.io.read_epochs_eeglab(row["file_path"], verbose="ERROR")
    epochs.load_data()
    epochs.pick(BANANA_CHANNELS)   # keep only banana channels

    data = epochs.get_data()          # (n_epochs, 4, n_times)
    sfreq = float(epochs.info["sfreq"])

    # per-epoch PTP rejection (consistent with notebook 02a)
    ptp_uv = np.ptp(data, axis=-1).max(axis=1) * 1e6  # (n_epochs,)
    keep   = np.where(ptp_uv <= REJECT_PTP_UV)[0]

    psds, freqs = mne.time_frequency.psd_array_welch(
        data[keep], sfreq=sfreq,
        fmin=1.0, fmax=40.0,
        n_fft=data.shape[-1], n_overlap=0,
        verbose="ERROR",
    )  # (n_kept, 4, n_freqs)

    for epoch_pos, orig_idx in enumerate(keep):
        feat = {}
        for band_name, (flo, fhi) in BANDS_BP.items():
            mask = (freqs >= flo) & (freqs < fhi)
            for ch_pos, ch_name in enumerate(BANANA_CHANNELS):
                feat[f"bp_{band_name}_{ch_name}"] = float(
                    np.log(np.maximum(psds[epoch_pos, ch_pos, mask].mean(), 1e-20))
                )
        bp_rows.append({
            "subject_id": row["subject_id"],
            "recording_number": int(row["recording_number"]),
            "epoch_index_original": int(orig_idx),
            "drug": row["drug"],
            **feat,
        })

bp_df = pd.DataFrame(bp_rows)
print(f"Bandpower rows: {len(bp_df)}  |  features: {len(bp_labels)}")
bp_df.to_csv(DERIVED_ROOT / "features" / "features_banana_bandpower.csv", index=False)
display(bp_df.head(3))

Bandpower rows: 276  |  features: 16


,subject_id,recording_number,epoch_index_original,drug,bp_delta_O1,bp_delta_O2,bp_delta_T7,bp_delta_T8,bp_theta_O1,bp_theta_O2,bp_theta_T7,bp_theta_T8,bp_alpha_O1,bp_alpha_O2,bp_alpha_T7,bp_alpha_T8,bp_beta_O1,bp_beta_O2,bp_beta_T7,bp_beta_T8
0,210,3,0,awake,-25.582074,-25.087534,-26.645024,-26.718708,-25.656751,-25.408495,-26.284728,-26.777134,-24.698336,-23.897596,-25.789848,-26.563586,-27.603186,-27.506469,-28.337232,-27.244932
1,210,3,1,awake,-25.382987,-25.180590,-26.670081,-26.966994,-24.599909,-24.364708,-25.459055,-26.353959,-24.070886,-23.730378,-26.263301,-26.523207,-27.170598,-27.178681,-28.828435,-27.558418
2,210,3,2,awake,-26.010287,-25.840860,-26.702079,-26.650626,-25.114308,-25.180842,-26.353396,-26.763147,-24.501014,-23.961398,-25.753874,-26.366604,-27.400821,-27.653462,-28.991228,-27.886853


In [5]:
# %%
# ============================================
# Section 2b. Load banana wPLI (B) from NPZ matrices
# ============================================

npz = np.load(MATRICES_PATH)
wpli_rows = []

for _, row in bp_df.iterrows():   # iterate over the same epochs kept after PTP rejection
    sid    = str(int(row["subject_id"]))
    recnum = int(row["recording_number"])
    eidx   = int(row["epoch_index_original"])

    edge_vec = []
    for band in BANDS_WPLI:
        key = f"{sid}__rec{recnum}__e{eidx:04d}__{band}"
        mat = npz[key]
        sub = mat[np.ix_(banana_idx, banana_idx)]
        edge_vec.append(sub[triu_r, triu_c])

    wpli_rows.append(dict(zip(wpli_labels, np.concatenate(edge_vec))))

wpli_df = pd.DataFrame(wpli_rows)
print(f"wPLI rows: {len(wpli_df)}  |  features: {len(wpli_labels)}")

wPLI rows: 276  |  features: 18


In [6]:
# %%
# ============================================
# Section 3. Build X, y, groups for all three variants
# ============================================

XA_ban = bp_df[bp_labels].to_numpy()
XB_ban = wpli_df[wpli_labels].to_numpy()
XC_ban = np.concatenate([XA_ban, XB_ban], axis=1)

y      = (bp_df["drug"] == "ketamine").astype(int).to_numpy()
groups = bp_df["subject_id"].astype(str).to_numpy()

from dataclasses import dataclass

@dataclass(frozen=True)
class Spec:
    X: np.ndarray
    name: str
    n_features: int

specs = [
    Spec(XA_ban, "A_banana_bandpower",  XA_ban.shape[1]),
    Spec(XB_ban, "B_banana_wpli",       XB_ban.shape[1]),
    Spec(XC_ban, "C_banana_A_plus_B",   XC_ban.shape[1]),
]

for s in specs:
    print(f"{s.name}: {s.X.shape}")

A_banana_bandpower: (276, 16)
B_banana_wpli: (276, 18)
C_banana_A_plus_B: (276, 34)


In [7]:
# %%
# ============================================
# Section 4. Nested CV RF — parallelized across (model × fold)
# No PCA: all feature sets are tiny (16, 18, 34)
# ============================================

param_dist = {
    "n_estimators":      randint(200, 1001),
    "max_depth":         [None, 5, 10, 20],
    "min_samples_split": randint(2, 11),
    "min_samples_leaf":  randint(1, 6),
    "max_features":      ["sqrt", "log2", None, 0.3, 0.5, 0.8],
}

outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
splits   = list(outer_cv.split(XA_ban, y, groups))
N_TASKS  = len(specs) * OUTER_SPLITS


def fit_eval_one(X: np.ndarray, model_name: str,
                 fold: int, tr: np.ndarray, te: np.ndarray) -> tuple:
    inner_cv = GroupKFold(n_splits=INNER_SPLITS)
    gtr, gte = groups[tr], groups[te]

    rs = RandomizedSearchCV(
        RandomForestClassifier(
            random_state=SEED, class_weight="balanced_subsample", n_jobs=1
        ),
        param_distributions=param_dist,
        n_iter=N_ITER, cv=inner_cv, scoring="balanced_accuracy",
        n_jobs=1, refit=True, random_state=SEED + fold, verbose=0,
    )
    rs.fit(X[tr], y[tr], groups=gtr)
    best  = rs.best_estimator_
    proba = best.predict_proba(X[te])[:, 1]
    yhat  = (proba >= 0.5).astype(int)

    try:
        auc = float(roc_auc_score(y[te], proba))
    except Exception:
        auc = np.nan

    cm = confusion_matrix(y[te], yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (np.nan,) * 4

    fold_row = {
        "model": model_name, "fold": fold,
        "n_test": int(len(te)), "n_test_subjects": int(len(np.unique(gte))),
        "accuracy": float(accuracy_score(y[te], yhat)),
        "balanced_accuracy": float(balanced_accuracy_score(y[te], yhat)),
        "roc_auc": auc,
        "tn": float(tn), "fp": float(fp), "fn": float(fn), "tp": float(tp),
    }
    pred_rows = [
        {"model": model_name, "fold": fold, "subject_id": gte[i],
         "y_true": int(y[te][i]), "y_proba": float(proba[i]), "y_pred": int(yhat[i])}
        for i in range(len(te))
    ]
    param_row = {
        "model": model_name, "fold": fold,
        "best_score_inner": float(rs.best_score_),
        **rs.best_params_,
    }
    return fold_row, pred_rows, param_row


tasks = [
    (s.X, s.name, fold, tr, te)
    for s in specs
    for fold, (tr, te) in enumerate(splits, start=1)
]

results = Parallel(n_jobs=min(N_CORES, N_TASKS), prefer="processes")(
    delayed(fit_eval_one)(*t) for t in tasks
)

folds_df  = pd.DataFrame([r[0] for r in results]).sort_values(["model", "fold"]).reset_index(drop=True)
preds_df  = pd.DataFrame([pr for r in results for pr in r[1]])
params_df = pd.DataFrame([r[2] for r in results])

print("Done.")
display(folds_df)

Done.


,model,fold,n_test,n_test_subjects,accuracy,balanced_accuracy,roc_auc,tn,fp,fn,tp
0,A_banana_bandpower,1,56,2,0.571429,0.571429,0.780612,6.0,22.0,2.0,26.0
1,A_banana_bandpower,2,55,2,0.690909,0.694444,0.787037,14.0,14.0,3.0,24.0
2,A_banana_bandpower,3,54,2,0.648148,0.631034,0.736552,25.0,4.0,15.0,10.0
3,A_banana_bandpower,4,56,2,0.535714,0.535714,0.688776,28.0,0.0,26.0,2.0
4,A_banana_bandpower,5,55,2,0.509091,0.509259,0.591270,14.0,14.0,13.0,14.0
5,B_banana_wpli,1,56,2,0.607143,0.607143,0.658163,14.0,14.0,8.0,20.0
6,B_banana_wpli,2,55,2,0.509091,0.507937,0.550265,16.0,12.0,15.0,12.0
7,B_banana_wpli,3,54,2,0.518519,0.510345,0.520000,18.0,11.0,15.0,10.0
8,B_banana_wpli,4,56,2,0.625000,0.625000,0.677296,19.0,9.0,12.0,16.0
9,B_banana_wpli,5,55,2,0.472727,0.470238,0.521164,17.0,11.0,18.0,9.0


In [8]:
# %%
# ============================================
# Section 5. Save results
# ============================================

folds_df.to_csv(OUT_DIR / "results_rf_banana_all.csv", index=False)
preds_df.to_csv(OUT_DIR / "predictions_rf_banana_all.csv", index=False)
params_df.to_csv(OUT_DIR / "best_params_rf_banana_all.csv", index=False)
print("Saved.")

Saved.


In [9]:
# %%
# ============================================
# Section 6. Full comparison table
# banana A / B / C  vs  full 62-ch A / B / C
# ============================================

full = pd.read_csv(FULL_RESULTS_PATH)

models_full = [
    ("A_bandpower_full62ch (26 feat)",   full[full["model"] == "A_bandpower_epoch"]),
    ("B_wpli_full62ch (5673 feat)",      full[full["model"] == "B_wpli_edges_epoch_sliding"]),
    ("C_A+B_full62ch (5699 feat)",       full[full["model"] == "C_AplusB_epoch_sliding"]),
]
models_banana = [
    (f"A_banana (16 feat)",  folds_df[folds_df["model"] == "A_banana_bandpower"]),
    (f"B_banana (18 feat)",  folds_df[folds_df["model"] == "B_banana_wpli"]),
    (f"C_banana (34 feat)",  folds_df[folds_df["model"] == "C_banana_A_plus_B"]),
]

rows = []
for label, df_m in models_full + models_banana:
    rows.append({
        "model": label,
        "ch_set": "full 62ch" if "full" in label else "banana 4ch",
        "bacc_mean": round(float(df_m["balanced_accuracy"].mean()), 4),
        "bacc_std":  round(float(df_m["balanced_accuracy"].std()),  4),
        "auc_mean":  round(float(df_m["roc_auc"].mean()),            4),
        "auc_std":   round(float(df_m["roc_auc"].std()),             4),
        "acc_mean":  round(float(df_m["accuracy"].mean()),           4),
    })

comparison = pd.DataFrame(rows)
print("\n=== Full comparison ===")
display(comparison.set_index("model"))


=== Full comparison ===


,ch_set,bacc_mean,bacc_std,auc_mean,auc_std,acc_mean
model,,,,,,
A_bandpower_full62ch (26 feat),full 62ch,0.7057,0.1481,0.8214,0.1586,0.7038
B_wpli_full62ch (5673 feat),full 62ch,0.5294,0.0792,0.5355,0.0682,0.5290
C_A+B_full62ch (5699 feat),full 62ch,0.6380,0.1213,0.7555,0.0936,0.6385
A_banana (16 feat),banana 4ch,0.5884,0.0748,0.7168,0.0805,0.5911
B_banana (18 feat),banana 4ch,0.5441,0.0679,0.5854,0.0764,0.5465
C_banana (34 feat),banana 4ch,0.6251,0.1206,0.7231,0.0796,0.6278


In [10]:
# %%
# ============================================
# Section 7. Plot 1 — per-fold balanced accuracy, all 6 models
# ============================================

plot_specs = [
    # (label, values, color, linestyle)
    ("A full",     full[full["model"]=="A_bandpower_epoch"]["balanced_accuracy"].values,         "steelblue", "-"),
    ("B full",     full[full["model"]=="B_wpli_edges_epoch_sliding"]["balanced_accuracy"].values, "coral",     "-"),
    ("C full",     full[full["model"]=="C_AplusB_epoch_sliding"]["balanced_accuracy"].values,     "purple",    "-"),
    ("A banana",   folds_df[folds_df["model"]=="A_banana_bandpower"]["balanced_accuracy"].values,  "steelblue", "--"),
    ("B banana",   folds_df[folds_df["model"]=="B_banana_wpli"]["balanced_accuracy"].values,       "coral",     "--"),
    ("C banana",   folds_df[folds_df["model"]=="C_banana_A_plus_B"]["balanced_accuracy"].values,   "purple",    "--"),
]

fig, ax = plt.subplots(figsize=(12, 5))
for i, (label, vals, color, ls) in enumerate(plot_specs):
    jitter = (np.random.RandomState(i).rand(len(vals)) - 0.5) * 0.10
    ax.scatter(np.full(len(vals), i) + jitter, vals, color=color, s=55,
               zorder=3, alpha=0.9, marker="o" if ls == "-" else "s")
    ax.hlines(vals.mean(), i - 0.22, i + 0.22, colors=color,
               linewidths=2.5, linestyles=ls, label=label)

ax.axhline(0.5, linestyle=":", color="grey", linewidth=1)
ax.set_xticks(range(len(plot_specs)))
ax.set_xticklabels([p[0] for p in plot_specs], fontsize=10, rotation=15, ha="right")
ax.set_ylabel("Balanced accuracy (outer fold)", fontsize=12)
ax.set_title("Per-fold balanced accuracy: full 62-ch vs. banana 4-ch (O1,O2,T7,T8)\n"
             "Solid = full | Dashed = banana", fontsize=12)
ax.set_ylim(0.35, 1.0)
ax.legend(fontsize=9, ncol=2)
fig.tight_layout()

out = FIGURE_DIR / "banana_all_models_bacc.png"
fig.savefig(out, dpi=150)
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/banana_all_models_bacc.png


In [11]:
# %%
# ============================================
# Section 8. Plot 2 — grouped bar: balanced accuracy by model family
# ============================================

families = [("A", "steelblue"), ("B", "coral"), ("C", "purple")]
x = np.arange(len(families))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))

for offset, (ch_set, hatch, label_str) in enumerate([
    ("full 62ch", "",   "full 62ch"),
    ("banana 4ch", "//", "banana 4ch"),
]):
    means, errs = [], []
    for fam, _ in families:
        row = comparison[
            comparison["model"].str.startswith(fam) &
            (comparison["ch_set"] == ch_set)
        ]
        means.append(float(row["bacc_mean"].values[0]))
        errs.append(float(row["bacc_std"].values[0]))
    ax.bar(
        x + (offset - 0.5) * width, means, width,
        color=[c for _, c in families], alpha=0.75 if offset == 0 else 0.45,
        hatch=hatch, edgecolor="black", linewidth=0.7,
        yerr=errs, capsize=4,
        label=label_str,
    )

ax.axhline(0.5, linestyle="--", color="grey", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(["A (bandpower)", "B (wPLI)", "C (A+B)"], fontsize=12)
ax.set_ylabel("Balanced accuracy (mean ± std)", fontsize=12)
ax.set_title("Impact of channel reduction to banana subset\n(solid = full 62ch, hatched = 4ch)", fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0.4, 1.0)
fig.tight_layout()

out = FIGURE_DIR / "banana_all_models_grouped_bar.png"
fig.savefig(out, dpi=150)
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/banana_all_models_grouped_bar.png


In [12]:
# %%
print("\n=== Summary ===")
display(comparison.set_index("model"))

print("\nFigures:")
for f in sorted(FIGURE_DIR.glob("banana_*.png")):
    print(" ", f.name)


=== Summary ===


,ch_set,bacc_mean,bacc_std,auc_mean,auc_std,acc_mean
model,,,,,,
A_bandpower_full62ch (26 feat),full 62ch,0.7057,0.1481,0.8214,0.1586,0.7038
B_wpli_full62ch (5673 feat),full 62ch,0.5294,0.0792,0.5355,0.0682,0.5290
C_A+B_full62ch (5699 feat),full 62ch,0.6380,0.1213,0.7555,0.0936,0.6385
A_banana (16 feat),banana 4ch,0.5884,0.0748,0.7168,0.0805,0.5911
B_banana (18 feat),banana 4ch,0.5441,0.0679,0.5854,0.0764,0.5465
C_banana (34 feat),banana 4ch,0.6251,0.1206,0.7231,0.0796,0.6278



Figures:
  banana_all_models_bacc.png
  banana_all_models_grouped_bar.png
  banana_comparison_all_metrics.png
  banana_comparison_bacc.png
  banana_feature_importances.png
